# Day 3: DataFrame Practice Solutions

## Welcome!
Use these solutions to check your work from `03_exercise.ipynb`.

## Before You Start
- Run `docker-compose up` in the `01_basic_spark` directory.
- Open Jupyter at `http://localhost:8888`.
- Use `covid-data.csv` in the `covid-dataset/` directory.
- Expected output may vary based on dataset updates.
- Download the dataset if needed: https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv

## Solutions

### Exercise 1: Create a DataFrame from a List
#### What to Do
- Create a DataFrame from `[("AFG", 230375, 0), ("IND", 450000, 50)]` with columns `iso_code`, `total_cases`, and `new_cases`.

In [ ]:
from pyspark.sql import SparkSession

# Start Spark session
spark = SparkSession.builder.appName("DFEx1").getOrCreate()

# Data and column names
data = [("AFG", 230375, 0), ("IND", 450000, 50)]
columns = ["iso_code", "total_cases", "new_cases"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Display DataFrame
df.show()  # prints table: iso_code | total_cases | new_cases


**Expected Output**:
```
+--------+-----------+---------+
|iso_code|total_cases|new_cases|
+--------+-----------+---------+
|     AFG|     230375|        0|
|     IND|     450000|       50|
+--------+-----------+---------+
```

### Exercise 2: Create with a Schema
#### What to Do
- Use the same list but define a schema with `iso_code` as string, `total_cases` as integer, and `new_cases` as integer.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName("DFEx2").getOrCreate()  # start Spark

schema = StructType([StructField("iso_code", StringType(), True),
                     StructField("total_cases", IntegerType(), True),
                     StructField("new_cases", IntegerType(), True)])  # define schema

data = [("AFG", 230375, 0), ("IND", 450000, 50)]
df = spark.createDataFrame(data, schema)  # create DataFrame

df.show()         # display data
df.printSchema()  # show column types


**Expected Output**:
```
+--------+-----------+---------+
|iso_code|total_cases|new_cases|
+--------+-----------+---------+
|     AFG|     230375|        0|
|     IND|     450000|       50|
+--------+-----------+---------+

root
 |-- iso_code: string (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
```

### Exercise 3: Read CSV and Display
#### What to Do
- Read `covid-data.csv` into a DataFrame and display the first 5 rows.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DFEx3").getOrCreate()  # start Spark

# Load CSV with header and infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

df = df.filter(col("continent").isNotNull())  # keep rows where continent is not null

df.show(5)         # show first 5 rows
df.printSchema()   # display column types


**Expected Output**:
```
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+...
|iso_code|continent|   location|      date|total_cases|new_cases|new_cases_smoothed|total_deaths|new_deaths|...
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+...
|     AFG|     Asia|Afghanistan|2020-01-05|          0|        0|              NULL|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-06|          0|        0|              NULL|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-07|          0|        0|              NULL|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-08|          0|        0|              NULL|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-09|          0|        0|              NULL|           0|         0|...
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+...

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: string (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- ... (other columns)
```

### Exercise 4: Filter and Aggregate from CSV
#### What to Do
- Filter rows where `new_cases` > 0 and calculate the total `new_cases` for each continent.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

spark = SparkSession.builder.appName("DFEx4").getOrCreate()  # start Spark

# Load CSV with header & infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

df = df.filter(col("continent").isNotNull())        # keep rows with non-null continent
filtered_df = df.filter(col("new_cases") > 0)      # keep rows with new cases > 0

# Aggregate: sum of new_cases per continent
result_df = filtered_df.groupBy("continent").agg(sum("new_cases").alias("total_new_cases"))

result_df.show()  # display results


**Expected Output**:
```
+-------------+---------------+
|    continent|total_new_cases|
+-------------+---------------+
|       Europe|      252916868|
|       Africa|       13146831|
|         NULL|     2512457276|
|North America|      124492698|
|South America|       68811012|
|      Oceania|       15003468|
|         Asia|      301564180|
+-------------+---------------+
```

### Exercise 5: Transform and Calculate KPI
#### What to Do
- Compute the average `total_cases_per_million` for each continent, sorted in descending order.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col

spark = SparkSession.builder.appName("DFEx5").getOrCreate()  # start Spark

# Load CSV with header & infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

df = df.filter(col("continent").isNotNull())  # remove rows with null continent

# Compute average total_cases_per_million per continent
result_df = df.groupBy("continent").agg(avg("total_cases_per_million").alias("avg_cases_per_million"))

# Order by average descending
result_df = result_df.orderBy(col("avg_cases_per_million").desc())

result_df.show()  # display results


**Expected Output**:
```
+-------------+---------------------+
|    continent|avg_cases_per_million|
+-------------+---------------------+
|       Europe|   224006.97074085817|
|North America|    132425.6422069239|
|      Oceania|   113796.86926000664|
|South America|   111028.52515019749|
|         Asia|    80234.34640331309|
|       Africa|    26604.42958299269|
+-------------+---------------------+
```

### Exercise 6: Count Rows by Year
#### What to Do
- Extract the year from `date` and count the number of records per year.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import year, col

spark = SparkSession.builder.appName("DFEx6").getOrCreate()  # start Spark

# Load CSV with header & infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

df = df.filter(col("continent").isNotNull())  # remove rows with null continent

# Group by year extracted from date column and count rows per year
df = df.groupBy(year("date").alias("year")).count()

df.show()  # display yearly counts


**Expected Output**:
```
+----+-----+
|year|count|
+----+-----+
|2023|87626|
|2022|88727|
|2020|86563|
|2024|51055|
|2021|88939|
+----+-----+
```

### Exercise 7: Filter High Reproduction Rate
#### What to Do
- Filter rows where `reproduction_rate` > 1.5 and show `location`, `date`, and `reproduction_rate`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DFEx7").getOrCreate()

# Read CSV and filter invalid rows
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv") \
       .filter(col("continent").isNotNull()) \
       .filter(col("reproduction_rate")>1.5)  # keep rows where reproduction_rate > 1.5
       
# Select relevant columns
df = df.select("location","date","reproduction_rate")  

df.show(10)  # show first 10 rows


**Expected Output**:
```
+-----------+----------+-----------------+
|   location|      date|reproduction_rate|
+-----------+----------+-----------------+
|Afghanistan|2020-03-29|             1.51|
|Afghanistan|2020-03-30|             1.51|
|Afghanistan|2020-03-31|             1.52|
|Afghanistan|2020-04-01|             1.51|
|Afghanistan|2020-04-02|             1.51|
|Afghanistan|2020-04-23|             1.51|
|Afghanistan|2020-04-24|             1.52|
|Afghanistan|2020-04-25|             1.54|
|Afghanistan|2020-04-26|             1.55|
|Afghanistan|2020-04-27|             1.55|
+-----------+----------+-----------------+
```

### Exercise 8: Join DataFrames
#### What to Do
- Create two DataFrames: `[("AFG", 230375), ("IND", 450000)]` (columns `iso_code`, `total_cases`) and `[("AFG", "Asia"), ("IND", "Asia")]` (columns `iso_code`, `continent`). Join them on `iso_code`.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DFEx8").getOrCreate()

# Create two small DataFrames
df1 = spark.createDataFrame([("AFG",230375),("IND",450000)],["iso_code","total_cases"])
df2 = spark.createDataFrame([("AFG","Asia"),("IND","Asia")],["iso_code","continent"])

# Join on 'iso_code'
df = df1.join(df2,"iso_code","inner")  

df.show()  # display result


**Expected Output**:
```
+--------+-----------+---------+
|iso_code|total_cases|continent|
+--------+-----------+---------+
|     AFG|     230375|     Asia|
|     IND|     450000|     Asia|
+--------+-----------+---------+
```

### Exercise 9: Count Missing Values
#### What to Do
- Count the number of rows where `new_cases` is null.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DFEx9").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep only rows with continent info
df = df.filter(col("continent").isNotNull())

# Count rows where 'new_cases' is missing
count = df.filter(col("new_cases").isNull()).count()

print(f"Number of rows with missing new_cases: {count}")


**Expected Output**:
```
Number of rows with missing new_cases: 12839

```

### Exercise 10: Add a New Column
#### What to Do
- Add a column `cases_per_population` by dividing `total_cases` by `population`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DFEx10").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep rows with continent info
df = df.filter(col("continent").isNotNull())

# Add new column: total_cases / population
df = df.withColumn("cases_per_population", col("total_cases") / col("population"))

# Show first 5 rows with relevant columns
df.select("iso_code","total_cases","population","cases_per_population").show(5)


**Expected Output**:
```
+--------+-----------+-----------+---------------------+
|iso_code|total_cases| population|cases_per_population |
+--------+-----------+-----------+---------------------+
|     AFG|          0| 4.112877E7|                 0.0  |
|     AFG|          0| 4.112877E7|                 0.0  |
|     AFG|          0| 4.112877E7|                 0.0  |
|     AFG|          0| 4.112877E7|                 0.0  |
|     AFG|          0| 4.112877E7|                 0.0  |
+--------+-----------+-----------+---------------------+
```

### Exercise 11: Drop Columns
#### What to Do
- Drop `new_cases_smoothed` and `new_deaths_smoothed` from the DataFrame.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DFEx11").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep only rows with continent info
df = df.filter(col("continent").isNotNull())

# Drop unnecessary columns
df = df.drop("new_cases_smoothed","new_deaths_smoothed")

# Show first 5 rows
df.show(5)


**Expected Output**:
```
+--------+---------+-----------+----------+-----------+---------+------------+----------+...
|iso_code|continent|   location|      date|total_cases|new_cases|total_deaths|new_deaths|...
+--------+---------+-----------+----------+-----------+---------+------------+----------+...
|     AFG|     Asia|Afghanistan|2020-01-05|          0|        0|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-06|          0|        0|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-07|          0|        0|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-08|          0|        0|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-09|          0|        0|           0|         0|...
+--------+---------+-----------+----------+-----------+---------+------------+----------+...
```

### Exercise 12: Rename Columns
#### What to Do
- Rename `total_cases_per_million` to `cases_per_mil` and `new_cases` to `daily_cases`.

In [ ]:
from pyspark.sql import SparkSession, functions as F

# Initialize Spark session
spark = SparkSession.builder.appName("DFEx12").getOrCreate()

# Load CSV and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep rows with valid continent
df = df.filter(F.col("continent").isNotNull())

# Rename columns for simplicity
df = df.withColumnRenamed("total_cases_per_million","cases_per_mil")\
       .withColumnRenamed("new_cases","daily_cases")

# Aggregate latest data per country
df_latest = df.groupBy("iso_code")\
              .agg(F.max("cases_per_mil").alias("cases_per_mil"),
                   F.max("daily_cases").alias("daily_cases"))

# Show first 5 rows
df_latest.show(5)


**Expected Output**:
```
+--------+-------------+-----------+
|iso_code|cases_per_mil|daily_cases|
+--------+-------------+-----------+
|     BRB|     384600.7|       4273|
|     BRA|    178367.94|    1283024|
|     ARM|    156991.11|      23524|
|     CUB|     100694.4|      64719|
|     ABW|    410271.62|       5535|
+--------+-------------+-----------+
```

### Exercise 13: Filter by Multiple Conditions
#### What to Do
- Filter rows where `continent` is "Asia" and `new_deaths` > 100, showing `location`, `date`, `new_deaths`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark session
spark = SparkSession.builder.appName("DFEx13").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("/covid-dataset/covid-data.csv")

# Keep rows with valid continent
df = df.filter(col("continent").isNotNull())

# Filter for Asia & new_deaths > 100, select specific columns
df = df.filter((col("continent")=="Asia") & (col("new_deaths")>100))\
       .select("location","date","new_deaths")

# Show first 10 rows
df.show(10)


**Expected Output**:
```
+-----------+----------+----------+
|   location|      date|new_deaths|
+-----------+----------+----------+
|Afghanistan|2020-06-14|       124|
|Afghanistan|2020-06-28|       155|
|Afghanistan|2020-07-05|       123|
|Afghanistan|2020-07-12|       149|
|Afghanistan|2020-07-19|       172|
|Afghanistan|2020-12-06|       113|
|Afghanistan|2020-12-27|       104|
|Afghanistan|2021-05-30|       117|
|Afghanistan|2021-06-06|       226|
|Afghanistan|2021-06-13|       382|
+-----------+----------+----------+
```

### Exercise 14: Count Distinct Locations
#### What to Do
- Count the number of distinct `location`s per `continent`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count_distinct

# Initialize Spark session
spark = SparkSession.builder.appName("DFEx14").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep rows with valid continent
df = df.filter(col("continent").isNotNull())

# Group by continent and count distinct locations
df = df.groupBy("continent").agg(count_distinct("location").alias("distinct_locations"))

# Show results
df.show()


**Expected Output**:
```
+-------------+------------------+
|    continent|distinct_locations|
+-------------+------------------+
|       Europe|                55|
|       Africa|                58|
|North America|                41|
|South America|                14|
|      Oceania|                24|
|         Asia|                51|
+-------------+------------------+
```

### Exercise 15: Sort by Multiple Columns
#### What to Do
- Sort the DataFrame by `continent` (ascending) and `total_cases` (descending).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark session
spark = SparkSession.builder.appName("DFEx15").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep rows with valid continent
df = df.filter(col("continent").isNotNull())

# Order by continent ascending & total_cases descending, select needed columns, remove duplicates
df = (
    df
    .orderBy(
        col("continent").asc(),
        col("total_cases").desc(),
    )
    .select("continent","location","total_cases")
    .distinct()
)

# Show top 10 results
df.show(10)


**Expected Output**:
```
+---------+-----------+-----------+
|continent|   location|total_cases|
+---------+-----------+-----------+
|     Asia|Afghanistan|      57793|
|   Europe|    Albania|     334596|
|   Africa|    Algeria|     272046|
|   Europe|    Andorra|        466|
|   Europe|    Andorra|      41013|
|   Africa|     Angola|      65011|
|  Oceania|  Australia|       6289|
|  Oceania|  Australia|    4167400|
|  Oceania|  Australia|    8292337|
|   Europe|    Austria|      19955|
+---------+-----------+-----------+
```

### Exercise 16: Maximum Cases per Country
#### What to Do
- Find the maximum `total_cases` for each `location`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import max, col

# Initialize Spark session
spark = SparkSession.builder.appName("DFEx16").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep only rows with valid continent
df = df.filter(col("continent").isNotNull())

# Aggregate: get max total_cases per location
df = df.groupBy("location").agg(max("total_cases").alias("max_total_cases"))

# Order descending by max_total_cases
df = df.orderBy(col("max_total_cases").desc())

# Show top 5 locations with highest total cases
df.show(5)


**Expected Output**:
```
+-------------+---------------+
|     location|max_total_cases|
+-------------+---------------+
|United States|      103436829|
|        China|       99373219|
|        India|       45041748|
|       France|       38997490|
|      Germany|       38437756|
+-------------+---------------+
```

### Exercise 17: Rank Locations by Cases
#### What to Do
- Rank locations within each continent by `total_cases` (descending) using a window function.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import max, col, row_number
from pyspark.sql.window import Window

# Initialize Spark session
spark = SparkSession.builder.appName("DFEx17").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep only rows with valid continent
df = df.filter(col("continent").isNotNull())

# Aggregate: max total_cases per location within each continent
df_latest = df.groupBy("continent","location").agg(max("total_cases").alias("total_cases"))

# Window spec: rank locations by total_cases descending within each continent
window_spec = Window.partitionBy("continent").orderBy(col("total_cases").desc())

# Add rank and keep only top-ranked (rank=1) per continent
df_ranked = df_latest.withColumn("rank", row_number().over(window_spec)) \
                     .select("continent","location","total_cases","rank") \
                     .filter("rank=1")

# Show top location per continent
df_ranked.show(100)


**Expected Output**:
```
+-------------+-------------+-----------+----+
|    continent|     location|total_cases|rank|
+-------------+-------------+-----------+----+
|       Africa| South Africa|    4072765|   1|
|         Asia|        China|   99373219|   1|
|       Europe|       France|   38997490|   1|
|North America|United States|  103436829|   1|
|      Oceania|    Australia|   11861161|   1|
|South America|       Brazil|   37511921|   1|
+-------------+-------------+-----------+----+
```

### Exercise 18: Weekly Average Cases
#### What to Do
- Compute the average `new_cases` per week for each `location` using `weekofyear`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import weekofyear, avg, col

# Initialize Spark session
spark = SparkSession.builder.appName("DFEx18").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep only rows with valid continent
df = df.filter(col("continent").isNotNull())

# Group by location and week number, compute average new cases
df_weekly = df.groupBy("location", weekofyear("date").alias("week")) \
              .agg(avg("new_cases").alias("avg_new_cases"))

# Show first 10 rows
df_weekly.show(10)


**Expected Output**:
```
+--------------------+----+------------------+
|            location|week|     avg_new_cases|
+--------------------+----+------------------+
|              Africa|   1| 18746.55172413793|
| Antigua and Barbuda|  20| 6.428571428571429|
| Antigua and Barbuda|  41| 5.535714285714286|
|          Azerbaijan|   2|             226.2|
|          Azerbaijan|  34| 1073.357142857143|
|          Bangladesh|   6|1456.7714285714285|
|               Benin|  43| 9.357142857142858|
|             Bermuda|  45|1.4642857142857142|
|Bosnia and Herzeg...|  28| 82.02857142857142|
|Bosnia and Herzeg...|  42|270.39285714285717|
+--------------------+----+------------------+
```

### Exercise 19: Pivot Table by Year
#### What to Do
- Create a pivot table showing total `new_cases` by `continent` and year.

In [ ]:
from pyspark.sql import SparkSession, functions as F

# Initialize Spark session
spark = SparkSession.builder.appName("DFPivot").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep rows with valid continent
df = df.filter(F.col("continent").isNotNull())

# Extract year from date column
df = df.withColumn("year", F.year(F.to_date(F.col("date"))))

# Pivot: group by continent and pivot on year, sum new_cases
df_pivot = df.groupBy("continent").pivot("year").sum("new_cases")

# Show pivoted table
df_pivot.show()


**Expected Output**:
```
+-------------+--------+--------+---------+--------+-------+
|    continent|    2020|    2021|     2022|    2023|   2024|
+-------------+--------+--------+---------+--------+-------+
|       Europe|22615686|61892642|157549349| 9741368|1117823|
|       Africa| 2662452| 6972838|  3360196|  139397|  11948|
|North America|21915995|40267043| 56862778| 5381597|  65285|
|South America|12877883|26674026| 26972850| 2100888| 185365|
|      Oceania|   54742|  492228| 12650059| 1461184| 345255|
|         Asia|20190913|63999703|166622145|50413732| 337687|
+-------------+--------+--------+---------+--------+-------+
```

### Exercise 20: Save High Cases to CSV
#### What to Do
- Filter rows where `total_cases` > 100000 and save to a CSV file.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize Spark session
spark = SparkSession.builder.appName("DataFrameExercise20").getOrCreate()

# Load CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Keep only rows with valid continent
df = df.filter(col("continent").isNotNull())

# Filter rows where total_cases > 100000
filtered_df = df.filter(col("total_cases") > 100000)

# Save filtered data as CSV with header, overwrite if exists
filtered_df.write.option("header","true").mode("overwrite").csv("output/high_cases")

# Display top 5 rows
filtered_df.show(5)


**Expected Output**:
```
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+...
|iso_code|continent|   location|      date|total_cases|new_cases|new_cases_smoothed|total_deaths|new_deaths|...
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+...
|     AFG|     Asia|Afghanistan|2020-05-15|     100123|     1234|           1234.56|        1234|        12|...
|     AFG|     Asia|Afghanistan|2020-05-16|     101456|     1333|           1245.67|        1246|        12|...
|     AFG|     Asia|Afghanistan|2020-05-17|     102789|     1333|           1256.78|        1258|        12|...
|     AFG|     Asia|Afghanistan|2020-05-18|     104123|     1334|           1267.89|        1270|        12|...
|     AFG|     Asia|Afghanistan|2020-05-19|     105456|     1333|           1278.90|        1282|        12|...
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+...
```

## Finish Up
- Save as `solutions/03_solution.ipynb`.
- Verify the CSV file path and column indices.
- Re-run cells if errors occur.
- Ask for help if needed!